In [2]:
# Write images from a folder into a GIF or MP4.
# GIF_START/GIF_END are list indices after sorting, using Python's [start:end] convention.
from pathlib import Path
from PIL import Image
import cv2
import numpy as np

GIF_IMAGES_DIR = "./viz_vehicle/Steering_VehicleOnRoad_1086_route0_05_27_18_23_29"
GIF_START = 5
GIF_END = 150
GIF_FPS = 10
mp4_mode = True
OUTPUT_PATH = Path("./gifs") / f"vehicle_stopping.{'mp4' if mp4_mode else 'gif'}"

def frame_idx_from_path(path):
    try:
        return int(Path(path).stem)
    except ValueError:
        return None

def image_sort_key(path):
    frame_idx = frame_idx_from_path(path) if "frame_idx_from_path" in globals() else None
    return (0, frame_idx) if frame_idx is not None else (1, path.name)


def list_images(image_dir):
    image_dir = Path(image_dir)
    image_paths = []
    for suffix in ("*.png", "*.jpg", "*.jpeg"):
        image_paths.extend(image_dir.glob(suffix))
    return sorted(image_paths, key=image_sort_key)


image_paths = list_images(GIF_IMAGES_DIR)
selected_paths = image_paths[GIF_START:GIF_END]

if not selected_paths:
    raise FileNotFoundError(
        f"No images found in slice [{GIF_START}:{GIF_END}] under {Path(GIF_IMAGES_DIR).resolve()}"
    )

frames = []
for path in selected_paths:
    with Image.open(path) as image:
        frames.append(image.convert("RGB").resize((1024//2, 1408//2)))

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
if mp4_mode:
    width, height = frames[0].size
    writer = cv2.VideoWriter(
        str(OUTPUT_PATH),
        cv2.VideoWriter_fourcc(*"mp4v"),
        GIF_FPS,
        (width, height),
    )
    if not writer.isOpened():
        raise RuntimeError(f"Could not open MP4 writer for {OUTPUT_PATH}")
    for frame in frames:
        writer.write(cv2.cvtColor(np.array(frame), cv2.COLOR_RGB2BGR))
    writer.release()
else:
    gif_frames = [frame.convert("P", palette=Image.ADAPTIVE) for frame in frames]
    gif_frames[0].save(
        OUTPUT_PATH,
        save_all=True,
        append_images=gif_frames[1:],
        duration=round(1000 / GIF_FPS),
        loop=0,
        optimize=False,
    )

print(f"{'mp4' if mp4_mode else 'gif'} frames: {len(frames)} from slice [{GIF_START}:{GIF_END}]")
print("saved:", OUTPUT_PATH.resolve())

mp4 frames: 145 from slice [5:150]
saved: /media/hcis-s20/SRL/osu_fail2drive/fail2drive/gifs/vehicle_stopping.mp4


In [ ]:
import json, math

alphas = ['0', '02', '04', '06', '08', '10']
for alpha in alphas:
    path = f"results/alpha{alpha}/1085_peds.json"

    with open(path) as f:
        data = json.load(f)

    record = data["_checkpoint"]["records"][0]
    scores = record["scores"]
    infractions = record["infractions"]

    ds = float(scores["score_composed"])
    rc = float(scores["score_route"])
    ip = float(scores["score_penalty"])

    ignored = {"min_speed_infractions", "outside_route_lanes"}
    success = all(
        not entries
        for name, entries in infractions.items()
        if name not in ignored
    )
    sr = 100.0 if success else 0.0

    hm = 0.0 if ds == 0 or sr == 0 else 2.0 / ((1.0 / ds) + (1.0 / sr))

    print(f"Alphas: {alpha}")
    # print(f"Status: {record['status']}")
    print(f"DS: {ds:.3f}")
    # print(f"RC: {rc:.3f}")
    # print(f"Infraction penalty: {ip:.6f}")
    print(f"SR: {sr:.1f}")
    print(f"HM: {hm:.3f}")

In [5]:
import json, math

alpha = '05'
path = f"results/alpha{alpha}/1085_peds.json"

with open(path) as f:
    data = json.load(f)

record = data["_checkpoint"]["records"][0]
scores = record["scores"]
infractions = record["infractions"]

ds = float(scores["score_composed"])
rc = float(scores["score_route"])
ip = float(scores["score_penalty"])

ignored = {"min_speed_infractions", "outside_route_lanes"}
success = all(
    not entries
    for name, entries in infractions.items()
    if name not in ignored
)
sr = 100.0 if success else 0.0

hm = 0.0 if ds == 0 or sr == 0 else 2.0 / ((1.0 / ds) + (1.0 / sr))

print(f"Alphas: {alpha}")
# print(f"Status: {record['status']}")
print(f"DS: {ds:.3f}")
# print(f"RC: {rc:.3f}")
# print(f"Infraction penalty: {ip:.6f}")
print(f"SR: {sr:.1f}")
print(f"HM: {hm:.3f}")

Alphas: 05
DS: 10.223
SR: 0.0
HM: 0.000
